<a href="https://colab.research.google.com/github/OLDHOUSE-MECHANIC/ColabNotebookTools-Optimized/blob/main/LTX_Vid_mini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install required libraries
!pip install -U diffusers transformers accelerate torch torchvision
# Install decord or opencv to help handle video saving if needed, but diffusers has a built-in helper
# Install the absolute latest cutting-edge version of diffusers and its requirements
!pip install -U git+https://github.com/huggingface/diffusers.git
!pip install -U transformers accelerate pipeline

  Cloning https://github.com/huggingface/diffusers.git to /tmp/pip-req-build-wujmhe5z
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/diffusers.git /tmp/pip-req-build-wujmhe5z
  Resolved https://github.com/huggingface/diffusers.git to commit a50ade492678dd105f3df1b3d3e0df35c4e0740e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for diffusers: filename=diffusers-0.39.0.dev0-py3-none-any.whl size=5576394 sha256=6fa34a38c3fb7343c7bb4bc47a6c7fb1b9ce8b6201fdb8992827d55fd339c0c1
  Stored in directory: /tmp/pip-ephem-wheel-cache-0g6equ3l/wheels/23/0f/7d/f97813d265ed0e599a78d83afd4e1925740896ca79b46cccfd
Successfully built diffusers
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.38.0
    Uninstalling diffusers-0.38.0:
      Successfully uninstalled diffusers-0.38.0


In [ ]:
import torch
from diffusers import DiffusionPipeline

model_id = "Lightricks/LTX-Video"

print("⏳ Loading unified LTX-Video pipeline...")
# DiffusionPipeline will automatically map to the optimized LTXPipeline
pipe = DiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    device_map="balanced"
)

print("✅ LTX-Video successfully loaded into GPU memory!")

⏳ Loading unified LTX-Video pipeline...


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

In [4]:
#@title 🎛️ LTX Anime Generation & Chaining
from google.colab import files
from PIL import Image
import io
import os
import torch

# --- COLAB FORM PARAMETERS ---
use_last_frame_of_previous_video = False #@param {type:"boolean"}
prompt = "Static camera, fixed isometric view, locked frame. Perfectly stable background with zero flickering, zero shifting, and absolutely no variations in colors, lighting, textures, or frames. The background environment remains 100% frozen and constant. Maintaining the exact 2D anime art style, the character of the young man in the blue shirt and white pants remains completely consistent in design as he wanders and paces around the empty room. Only the character moves; everything else is perfectly static." #@param {type:"string"}
negative_prompt = "photorealistic, 3d render, blurry, low quality, deformed faces, realistic texturing" #@param {type:"string"}

# LTX strict frame requirement: must follow (8 * n) + 1 rule (e.g., 17, 25, 33, 41, 49)
num_frames = 25 #@param {type:"integer"}
fps = 12 #@param {type:"integer"}
num_inference_steps = 30 #@param {type:"integer"}

# History routing
history_dir = "/content/generation_history"
os.makedirs(history_dir, exist_ok=True)
last_frame_history_path = os.path.join(history_dir, "last_frame_cache.png")

# --- 1. INITIAL FRAME SELECTION ---
init_image = None
if use_last_frame_of_previous_video and os.path.exists(last_frame_history_path):
    print("🔗 Chaining enabled: Loading the last frame from your previous anime clip...")
    init_image = Image.open(last_frame_history_path).convert("RGB")
else:
    print("📸 Please upload your INITIAL Anime frame...")
    uploaded_init = files.upload()
    if uploaded_init:
        init_filename = list(uploaded_init.keys())[0]
        init_image = Image.open(io.BytesIO(uploaded_init[init_filename])).convert("RGB")

if init_image is None:
    raise ValueError("❌ No initial image found. Please upload a file or enable chaining.")

# Ensure dimensions are multiples of 32 for the structural grid
init_image = init_image.resize((768, 512))

# --- 2. RUN GENERATION ---
print("\n🚀 Animating using LTX cond_image formatting...")
with torch.inference_mode():
    video_frames = pipe(
        cond_image=init_image,  # <-- Swapped 'image' to 'cond_image' here
        prompt=prompt,
        negative_prompt=negative_prompt,
        width=768,
        height=512,
        num_frames=num_frames,
        num_inference_steps=num_inference_steps,
    ).frames[0]

# --- 3. CACHE FINAL FRAME FOR NEXT CLIP ---
try:
    final_frame = video_frames[-1]
    final_frame.save(last_frame_history_path)
    print(f"💾 Cached the final anime frame for seamless sequential chaining!")
except Exception as e:
    print(f"⚠️ Could not cache final frame: {e}")

📸 Please upload your INITIAL Anime frame...


Saving Room Empty Box.jpeg to Room Empty Box (1).jpeg

🚀 Animating using DiffusionPipeline mapping...


TypeError: LTXPipeline.__call__() got an unexpected keyword argument 'image'

In [ ]:
#@title 💾 Export Video
from diffusers.utils import export_to_video
from ipywidgets import Video

output_video_path = "/content/generated_output.mp4" #@param {type:"string"}

export_to_video(video_frames, output_video_path, fps=fps)
Video.from_file(output_video_path, width=512, height=512)